websocket connection demo 

In [2]:
import json
import websockets

URL = "ws://2e4e907b.fast.gemini.com"

async def demo(n=10):
    async with websockets.connect(URL) as ws:
        await ws.send(json.dumps({"id":"1","method":"subscribe","params":["btcusd@bookTicker"]}))
        for _ in range(n):
            print(await ws.recv())

await demo(10)


{"u":1764525463505039,"E":1770956413439893956,"s":"btcusd","b":"66565.72000","B":"0.0162244500","a":"66566.32000","A":"0.0621339600"}
{"id":"1","status":200}
{"u":1764525463505232,"E":1770956413530390320,"s":"btcusd","b":"66565.72000","B":"0.0162244500","a":"66566.32000","A":"0.0227000000"}
{"u":1764525463507803,"E":1770956415510777546,"s":"btcusd","b":"66565.72000","B":"0.0389244500","a":"66566.32000","A":"0.0227000000"}
{"u":1764525463507804,"E":1770956415510781229,"s":"btcusd","b":"66565.72000","B":"0.0389244500","a":"66568.51000","A":"0.0200000000"}
{"u":1764525463507805,"E":1770956415510995366,"s":"btcusd","b":"66565.98000","B":"0.0227000000","a":"66568.51000","A":"0.0200000000"}
{"u":1764525463508259,"E":1770956415570124542,"s":"btcusd","b":"66565.98000","B":"0.0227000000","a":"66568.49000","A":"0.0227000000"}
{"u":1764525463508574,"E":1770956415626496658,"s":"btcusd","b":"66565.72000","B":"0.0389244500","a":"66568.49000","A":"0.0227000000"}
{"u":1764525463508576,"E":177095641562

Gemini public api:

In [ ]:
# Tradable symbols
import requests
BASE_URL = "https://api.gemini.com"  # or sandbox
symbols = requests.get(BASE_URL + "/v1/symbols", timeout=10).json()
symbols[:20]


['2zgusd',
 '2zrlusd',
 '2zusd',
 '2zusdc',
 'aavegusd',
 'aaverlusd',
 'aaveusd',
 'aaveusdc',
 'aligusd',
 'alirlusd',
 'aliusd',
 'aliusdc',
 'ampgusd',
 'amprlusd',
 'ampusd',
 'ampusdc',
 'ankrgusd',
 'ankrrlusd',
 'ankrusd',
 'ankrusdc']

In [6]:
#trade constraints
sym = "btcusd"
details = requests.get(BASE_URL + f"/v1/symbols/details/{sym}", timeout=10).json()
details


{'symbol': 'BTCUSD',
 'base_currency': 'BTC',
 'quote_currency': 'USD',
 'tick_size': 1e-08,
 'quote_increment': 0.01,
 'min_order_size': '0.00001',
 'status': 'open',
 'wrap_enabled': False,
 'product_type': 'spot',
 'contract_type': 'vanilla',
 'contract_price_currency': 'USD'}

In [7]:
t2 = requests.get(BASE_URL + f"/v2/ticker/{sym}", timeout=10).json()
t2


{'symbol': 'BTCUSD',
 'open': '67819.19',
 'high': '70070.88',
 'low': '67265.85',
 'close': '68414.94',
 'changes': ['67882.11',
  '67819.19',
  '67930.74',
  '67829.73',
  '67479.67',
  '68336.69',
  '68361.5',
  '69704.84',
  '68791.02',
  '68784.77',
  '68950.22',
  '68704.49',
  '68536.06',
  '68431.66',
  '68360.54',
  '68823.84',
  '68376.27',
  '68260.86',
  '68587.86',
  '68809.87',
  '68786.33',
  '68919.2',
  '68834.63',
  '68414.94'],
 'bid': '67883.57',
 'ask': '67894.87'}

In [18]:
#order book
book = requests.get(BASE_URL + f"/v1/book/{sym}", params={"limit_bids":10,"limit_asks":10}, timeout=10).json()
book["bids"][0], book["asks"][0]


({'price': '67842.96', 'amount': '0.021', 'timestamp': '1771272651'},
 {'price': '67844.73', 'amount': '0.03617245', 'timestamp': '1771272651'})

In [19]:
trades = requests.get(BASE_URL + f"/v1/trades/{sym}", params={"limit_trades":50}, timeout=10).json()
trades[0]


{'timestamp': 1771272650,
 'timestampms': 1771272650269,
 'tid': 2840140983569025,
 'price': '67844.79',
 'amount': '0.00014374',
 'exchange': 'gemini',
 'type': 'buy'}

In [38]:
candles = requests.get(BASE_URL + f"/v2/candles/{sym}/1m", timeout=10).json()
candles[0]  # array row


[1771272600000, 67864.86, 67913.45, 67844.39, 67844.79, 0.03053611]

Private API (we trade via this)

In [1]:
import requests

BASE_URL = "https://api.gemini.com"  # or sandbox

def get_json(path: str, params=None):
    r = requests.get(BASE_URL + path, params=params, timeout=15)
    r.raise_for_status()
    return r.json()

symbols = get_json("/v1/symbols")
print("num symbols:", len(symbols), "example:", symbols[:10])

# pick one symbol from /v1/symbols (Gemini symbols are typically lowercase like "btcusd")
sym = "btcusd"

ticker = get_json(f"/v1/pubticker/{sym}")
book   = get_json(f"/v1/book/{sym}", params={"limit_bids": 10, "limit_asks": 10})
candles = get_json(f"/v2/candles/{sym}/1m")

print("ticker:", ticker)
print("best bid/ask:", book["bids"][0], book["asks"][0])
print("first candle row:", candles[0] if candles else None)


num symbols: 447 example: ['2zgusd', '2zrlusd', '2zusd', '2zusdc', 'aavegusd', 'aaverlusd', 'aaveusd', 'aaveusdc', 'aligusd', 'alirlusd']
ticker: {'bid': '67876.74', 'ask': '67876.75', 'last': '67841.64', 'volume': {'BTC': '160.16386229', 'USD': '10865779.0864877556', 'timestamp': 1771272133000}}
best bid/ask: {'price': '67876.74', 'amount': '0.20866702', 'timestamp': '1771272133'} {'price': '67876.75', 'amount': '0.0114766', 'timestamp': '1771272133'}
first candle row: [1771272060000, 67856.16, 67856.16, 67841.64, 67841.64, 0.00012206]


In [3]:
import os, time, json, base64, hmac, hashlib, requests
from typing import Optional, Dict, Any

BASE_URL = "https://api.gemini.com"  # or sandbox
os.environ["GEMINI_API_KEY"] = "account-becUzqtHQIEmAkEQz9oA"      # put your key
os.environ["GEMINI_API_SECRET"] = "2DVi7YrDT4np721kMs9Q6yctnXeX"       # put your secret

API_KEY = os.environ["GEMINI_API_KEY"]
API_SECRET = os.environ["GEMINI_API_SECRET"].encode()

def gemini_private_post(path: str, payload_extra: Optional[Dict[str, Any]] = None):
    payload: Dict[str, Any] = {
        "request": path,
        "nonce": int(time.time() * 1000),  # ms; must increase per API key
    }
    if payload_extra:
        payload.update(payload_extra)

    encoded = json.dumps(payload).encode()
    b64 = base64.b64encode(encoded)              # bytes
    sig = hmac.new(API_SECRET, b64, hashlib.sha384).hexdigest()

    headers = {
        "Content-Type": "text/plain",
        "Content-Length": "0",
        "X-GEMINI-APIKEY": API_KEY,
        "X-GEMINI-PAYLOAD": b64.decode(),
        "X-GEMINI-SIGNATURE": sig,
        "Cache-Control": "no-cache",
    }

    r = requests.post(BASE_URL + path, headers=headers, timeout=15)
    r.raise_for_status()
    return r.json()


In [ ]:
balances = gemini_private_post("/v1/balances", {
    "showPendingBalances": True
})
print(balances[:2])


HTTPError: 400 Client Error: Bad Request for url: https://api.gemini.com/v1/balances

In [ ]:
order = gemini_private_post("/v1/order/new", {
    "symbol": "btcusd",
    "amount": "0.001",     # as string
    "price": "100.00",     # limit price (string)
    "side": "buy",
    "type": "exchange limit",  # common for spot limit orders
    # "options": ["immediate-or-cancel"]  # optional
})
print(order)
